# IMMapp — LayoutXLM fine-tuning pe FATURA

Acest notebook pregătește și antrenează modelul real pe un GPU Google Colab. Nu conține rezultate fabricate și nu modifică inferența locală până când arhiva modelului este importată explicit în IMMapp.

În Colab selectează **Runtime → Change runtime type → T4 GPU** înainte de a începe.

## 1. Verifică GPU-ul
Celula trebuie să afișeze un GPU NVIDIA. Dacă `nvidia-smi` lipsește, schimbă runtime-ul pe GPU.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU CUDA indisponibil. Activează T4 GPU din Runtime."
print("CUDA disponibil:", torch.cuda.get_device_name(0))

## 2. Montează Google Drive și configurează căile
Pune proiectul `immate-dash-pro` și `invoices_dataset_final.zip` în `MyDrive/IMMapp/`, sau editează căile de mai jos. Poți încărca arhivele direct în `/content` dacă nu folosești Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/IMMapp/immate-dash-pro')
DATASET_ZIP = Path('/content/drive/MyDrive/IMMapp/invoices_dataset_final.zip')

assert PROJECT_DIR.exists(), f'Proiectul nu există: {PROJECT_DIR}'
assert DATASET_ZIP.exists(), f'Datasetul nu există: {DATASET_ZIP}'
print('Proiect:', PROJECT_DIR)
print('Dataset:', DATASET_ZIP)

### Alternativă: încarcă arhivele direct
Dacă nu folosești Drive, decomentează `files.upload()`, încarcă ZIP-ul proiectului și datasetul, apoi extrage proiectul în `/content/immate-dash-pro`.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# !unzip -q /content/immate-dash-pro.zip -d /content
# PROJECT_DIR = Path('/content/immate-dash-pro')
# DATASET_ZIP = Path('/content/invoices_dataset_final.zip')

## 3. Instalează dependențele
Instalarea Detectron2 poate dura câteva minute. Mesajele de build sunt normale.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq git build-essential
%cd {PROJECT_DIR}
!python -m pip install -q --upgrade pip
!python -m pip install -q -r document-ai-backend/requirements-training.txt
!python -m pip install -q ninja cython pycocotools
!python -m pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!python -c "import torch, transformers, accelerate; print(torch.__version__, transformers.__version__)"

## 4. Copiază datasetul în locația standard
Fișierul mare rămâne în afara Git. Scripturile citesc ZIP-ul direct, fără extragerea celor ~40.000 de fișiere.

In [ ]:
import shutil
LOCAL_DATASET = PROJECT_DIR / 'document-ai-backend/datasets/fatura/invoices_dataset_final.zip'
LOCAL_DATASET.parent.mkdir(parents=True, exist_ok=True)
if DATASET_ZIP.resolve() != LOCAL_DATASET.resolve():
    shutil.copy2(DATASET_ZIP, LOCAL_DATASET)
print('Dataset pregătit:', LOCAL_DATASET)

## 5. Inspectează și pregătește FATURA
Rezultatul așteptat este 8.600 hugg valide și JSONL train/dev/test fără erori.

In [ ]:
%cd {PROJECT_DIR}
!python document-ai-backend/training/inspect_fatura_dataset.py "{LOCAL_DATASET}"
!python document-ai-backend/training/prepare_layoutxlm_dataset.py "{LOCAL_DATASET}"

## 6. Smoke-test pe GPU
Rulează doar două documente și un pas. Greutățile temporare sunt șterse automat.

In [ ]:
!python document-ai-backend/training/train_layoutxlm_invoice_classifier.py --smoke-test --device cuda

## 7. Antrenare completă
Comanda recomandată folosește 3 epoci și batch 2. Nu întrerupe runtime-ul. Pentru memorie redusă folosește comanda alternativă comentată.

In [ ]:
!python document-ai-backend/training/train_layoutxlm_invoice_classifier.py --device cuda --fp16 --epochs 3 --batch-size 2 --save

# Variantă low-memory (rulează aceasta în locul comenzii de mai sus):
# !python document-ai-backend/training/train_layoutxlm_invoice_classifier.py --device cuda --fp16 --epochs 2 --batch-size 1 --save

## 8. Arhivează și descarcă modelul
Rulează numai după ce antrenarea s-a terminat cu succes. Arhiva conține configurația, tokenizerul și greutățile HuggingFace.

In [ ]:
from google.colab import files
MODEL_DIR = PROJECT_DIR / 'document-ai-backend/models/layoutxlm-invoice-token-classifier'
assert (MODEL_DIR / 'config.json').exists(), 'Modelul nu pare salvat corect.'
archive_base = '/content/layoutxlm-invoice-token-classifier'
archive_path = shutil.make_archive(archive_base, 'zip', MODEL_DIR.parent, MODEL_DIR.name)
print('Arhivă creată:', archive_path)
files.download(archive_path)

## 9. După descărcare
Pe MacBook importă arhiva cu `npm run document-ai:import-trained-model -- /cale/layoutxlm-invoice-token-classifier.zip`, repornește `npm run dev`, verifică `/health`, apoi rulează benchmark-ul. Nu raporta rezultate de fine-tuning înainte de această evaluare.